In [1]:
import os
from pathlib import Path
import torch
from ultralytics import YOLO

print(f"CUDA available: {torch.cuda.is_available()}")

DATA_DIR = "fer2013"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

print(f"Train directory exists: {os.path.exists(TRAIN_DIR)}")
print(f"Test directory exists: {os.path.exists(TEST_DIR)}")

emotions = sorted([d.name for d in Path(TRAIN_DIR).iterdir() if d.is_dir()])
print(f"Emotion classes: {emotions}")

CUDA available: True
Train directory exists: True
Test directory exists: True
Emotion classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [3]:
import os

# --- UPDATE THIS TO YOUR MAIN RAF-DB FOLDER ---
# (The folder that contains the 'train' and 'test' subfolders)
DATASET_DIR = r'C:\Users\clogg\Desktop\YOLO\RAF-DB'
# ----------------------------------------------

emotion_map = {
    '1': 'surprise', '2': 'fear', '3': 'disgust', 
    '4': 'happy', '5': 'sad', '6': 'angry', '7': 'neutral'
}

print("Renaming folders...")

for split in ['train', 'test']:
    split_dir = os.path.join(DATASET_DIR, split)
    
    # Check if the train/test folder exists
    if not os.path.exists(split_dir):
        print(f"⚠️ Could not find folder: {split_dir}")
        continue

    for folder_num, emotion_name in emotion_map.items():
        old_path = os.path.join(split_dir, folder_num)
        new_path = os.path.join(split_dir, emotion_name)
        
        # If the numbered folder exists, rename it
        if os.path.exists(old_path):
            os.rename(old_path, new_path)
            print(f"✅ Renamed: {split}/{folder_num} -> {split}/{emotion_name}")
        elif os.path.exists(new_path):
            print(f"ℹ️ Already named correctly: {split}/{emotion_name}")

print("\n🚀 Phase 1 Complete! Move straight to the Phase 2 Preprocessing script.")

Renaming folders...
✅ Renamed: train/1 -> train/surprise
✅ Renamed: train/2 -> train/fear
✅ Renamed: train/3 -> train/disgust
✅ Renamed: train/4 -> train/happy
✅ Renamed: train/5 -> train/sad
✅ Renamed: train/6 -> train/angry
✅ Renamed: train/7 -> train/neutral
✅ Renamed: test/1 -> test/surprise
✅ Renamed: test/2 -> test/fear
✅ Renamed: test/3 -> test/disgust
✅ Renamed: test/4 -> test/happy
✅ Renamed: test/5 -> test/sad
✅ Renamed: test/6 -> test/angry
✅ Renamed: test/7 -> test/neutral

🚀 Phase 1 Complete! Move straight to the Phase 2 Preprocessing script.


In [5]:
from ultralytics import YOLO
import os

# Point this to the folder containing your 'train' and 'val' directories
DATA_PATH = r"C:\Users\clogg\Desktop\YOLO\RAF-DB" 

model = YOLO('yolov8m-cls.pt') 

results = model.train(
    data=DATA_PATH, 
    epochs=50, 
    imgsz=224, 
    batch=8,           # Lowered for laptop VRAM
    degrees=20.0,      # Rotates image randomly +/- 20 degrees
    shear=10.0,        # Distorts the image slightly
    perspective=0.001, # Adds a slight 3D tilt
    flipud=0.5,        # 50% chance to flip the image upside down
    project='Emotion_Project',
    name='YOLO_Augmented_Run'
)

print("Augmented Training Complete!")

New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\clogg\Desktop\YOLO\RAF-DB, degrees=20.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosa

In [7]:
import os
from ultralytics import YOLO
from sklearn.metrics import classification_report

# 1. Load your freshly trained model
# Make sure this points to the "best.pt" file YOLO just generated
model_path = r"C:\Users\clogg\Desktop\YOLO\runs\classify\Emotion_Project\YOLO_Augmented_Run2\weights\best.pt"
model = YOLO(model_path)

# 2. Point to your validation folder
val_dir = r"C:\Users\clogg\Desktop\YOLO\RAF-DB\test"

# YOLO automatically saved your class names inside the model
class_names = model.names # Looks like {0: 'angry', 1: 'disgust', ...}
target_names = [class_names[i] for i in range(len(class_names))]

y_true = []
y_pred = []

print("Running YOLO on validation images... (This will take about 15-30 seconds)")

# 3. Loop through the validation images and make predictions
for class_id, class_name in class_names.items():
    folder_path = os.path.join(val_dir, class_name)
    if not os.path.isdir(folder_path): continue
    
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        
        # Make a prediction (verbose=False stops it from spamming your console)
        results = model(img_path, verbose=False)
        
        # Extract the highest probability guess
        pred_class_id = results[0].probs.top1
        
        # Record the true answer and YOLO's guess
        y_true.append(class_id)
        y_pred.append(pred_class_id)

# 4. Print the final table!
print("\n" + "="*55)
print(" YOLOv8 CLASSIFICATION REPORT (BASELINE) ")
print("="*55)
report = classification_report(y_true, y_pred, target_names=target_names, digits=2)
print(report)

Running YOLO on validation images... (This will take about 15-30 seconds)

 YOLOv8 CLASSIFICATION REPORT (BASELINE) 
              precision    recall  f1-score   support

       angry       0.80      0.80      0.80       162
     disgust       0.73      0.72      0.72       160
        fear       0.71      0.64      0.67        74
       happy       0.95      0.92      0.94      1185
     neutral       0.83      0.86      0.84       680
         sad       0.83      0.85      0.84       478
    surprise       0.82      0.85      0.84       329

    accuracy                           0.87      3068
   macro avg       0.81      0.80      0.81      3068
weighted avg       0.87      0.87      0.87      3068

